# CT118-3-3 — Assignment 2: Model Building, Tuning & Evaluation
## Miami Housing Dataset

This notebook covers all four tasks of Assignment 2: Model Building (Task 1), Model Tuning (Task 2),
Model Evaluation & Discussion (Task 3), and Conclusion (Task 4). It builds directly on the outputs of
the Assignment 1 Data Preparation notebook.

**Before running:** place `X_train.csv`, `X_val.csv`, `X_test.csv`, `y_train.csv`, `y_val.csv`, and
`y_test.csv` (produced at the end of the Assignment 1 notebook) in the same folder as this notebook.

**Carried over from Assignment 1 — important context for every step below:**
- The target (`y_train`/`y_val`/`y_test`) is **`log(SALE_PRC)`**, not raw price, following the skewness
finding in Assignment 1 (3.22 → 0.74 after log transform). Every evaluation step below reports metrics
in **both** log-space and back-transformed dollar terms (via `np.expm1`), so results are interpretable.
- The feature set (`X_train` etc.) already has outlier capping and `StandardScaler` scaling applied,
fitted on training data only.
- The two candidate architectures were proposed in Assignment 1, Section 4: **Model A** (baseline ANN)
and **Model B** (deeper ANN with dropout and batch normalisation).


## Step 0: Import Libraries

We import TensorFlow/Keras for model building, plus the same evaluation and plotting libraries used in
Assignment 1. If `keras-tuner` is not yet installed, uncomment the pip install line and run it once.


In [ ]:
# !pip install keras-tuner --quiet

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import keras_tuner as kt

tf.random.set_seed(42)
np.random.seed(42)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)


## Step 1: Load Prepared Data from Assignment 1

We load the six CSV files produced at the end of the Assignment 1 notebook. Loading these directly
(rather than repeating the pre-processing pipeline here) guarantees that Assignment 2 uses **exactly**
the same cleaned, capped, and scaled data across every model, keeping the comparison fair.


In [ ]:
X_train = pd.read_csv('X_train.csv')
X_val = pd.read_csv('X_val.csv')
X_test = pd.read_csv('X_test.csv')
y_train = pd.read_csv('y_train.csv').squeeze()
y_val = pd.read_csv('y_val.csv').squeeze()
y_test = pd.read_csv('y_test.csv').squeeze()

print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_val:   {X_val.shape}, y_val:   {y_val.shape}")
print(f"X_test:  {X_test.shape}, y_test:  {y_test.shape}")

n_features = X_train.shape[1]
print(f"\nNumber of input features: {n_features}")


**Expected output / confirm:**
- Row counts should match Assignment 1's split exactly: **9,751 train / 2,091 validation / 2,090 test**.
- `n_features` should be **16** (the final feature set from Assignment 1: structural, locational,
environmental, temporal features, plus the engineered `AVG_CNTR_DIST`). If this number differs, double
check you copied the correct CSVs from Assignment 1's final notebook run.
- Remember: `y_train`/`y_val`/`y_test` are **log-transformed** prices, not raw dollars.


## Step 2: Build Model A — Baseline Artificial Neural Network

This is the shallow architecture proposed in Assignment 1, Section 4.1: two hidden layers (64 and 32
neurons) with ReLU activation, and a single linear output neuron for regression. Its purpose is to
establish whether a simple neural network can learn the price relationship at all, before adding
complexity in Model B.

**Design decision:** the output layer uses a single neuron with **no activation function** (linear),
which is standard for regression — applying an activation like ReLU or sigmoid here would incorrectly
constrain the range of predicted (log) prices.


In [ ]:
def build_model_a(input_dim):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(64, activation='relu'),
        layers.Dense(32, activation='relu'),
        layers.Dense(1)  # linear output for regression
    ], name='Model_A_Baseline')

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='mse',
        metrics=['mae']
    )
    return model

model_a = build_model_a(n_features)
model_a.summary()


**Expected output / note down:**
- The model summary should show 2 hidden `Dense` layers (64 and 32 units) plus a final `Dense(1)` output
layer, with a manageable total parameter count (a few thousand parameters — this is a small model by
design, matching its role as a baseline).
- Note the total parameter count for your report as a point of comparison against Model B's larger
parameter count later.


## Step 3: Train Model A

We train Model A using the Adam optimizer and Mean Squared Error (MSE) loss — the standard choice for
regression, since it directly penalises large prediction errors, which is desirable when predicting
price. `EarlyStopping` is used to halt training once validation loss stops improving, preventing
wasted epochs and overfitting.

**Design decision:** `EarlyStopping` with `patience=15` and `restore_best_weights=True` ensures the
final model corresponds to its best validation performance, not simply whatever epoch training happened
to stop at.


In [ ]:
early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=15, restore_best_weights=True
)

history_a = model_a.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=200,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)


**Expected output / note down:**
- Training should stop before reaching all 200 epochs (thanks to `EarlyStopping`) — note the actual
epoch it stopped at for your report.
- Watch the printed `loss` and `val_loss` per epoch: both should generally decrease and roughly track
each other. A large, persistent gap (val_loss much higher than loss) would indicate overfitting — note
whether this happens, since it directly informs the Task 3 discussion.


## Step 4: Build Model B — Deeper Regularised Artificial Neural Network

This is the deeper architecture from Assignment 1, Section 4.2: four hidden layers (128, 64, 32, 16
neurons) with batch normalisation and dropout after each hidden layer. The added depth is intended to
capture more complex non-linear interactions (e.g. between locational distance variables and structural
quality), while batch normalisation and dropout mitigate the higher overfitting risk that comes with
extra depth.

**Design decision:** dropout rate is set to **0.3** (a common, moderate starting point), and batch
normalisation is placed **before** the activation-bearing dense layer's next layer, which is standard
practice for stabilising training in deeper networks.


In [ ]:
def build_model_b(input_dim):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),

        layers.Dense(128, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),

        layers.Dense(64, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),

        layers.Dense(32, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.2),

        layers.Dense(16, activation='relu'),

        layers.Dense(1)  # linear output for regression
    ], name='Model_B_Deeper_Regularised')

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='mse',
        metrics=['mae']
    )
    return model

model_b = build_model_b(n_features)
model_b.summary()


**Expected output / note down:**
- The summary should show 4 hidden `Dense` layers interleaved with `BatchNormalization` and `Dropout`
layers, and a noticeably higher total parameter count than Model A (likely in the tens of thousands
range, driven mainly by the 128-neuron first layer).
- Note this parameter count for your report's Task 1 comparison table.


## Step 5: Train Model B

Same training setup as Model A (Adam, MSE loss, early stopping) for a fair comparison — the only
differences between the two models should be architectural, not training-procedure related.


In [ ]:
early_stop_b = keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=15, restore_best_weights=True
)

history_b = model_b.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=200,
    batch_size=32,
    callbacks=[early_stop_b],
    verbose=1
)


**Expected output / note down:**
- Again note the stopping epoch. Because of dropout and batch normalisation, Model B's `loss` may look
slightly noisier epoch-to-epoch than Model A's — this is expected and not itself a problem.
- Compare final `val_loss` between Model A and Model B here as a first, rough signal of which performed
better, before the formal evaluation in Step 6.


## Step 6: Plot Training vs Validation Loss Curves

We plot both models' loss curves side by side. This is one of the most important diagnostic plots in
the whole assignment — it directly shows overfitting (validation loss rising while training loss keeps
falling) or underfitting (both losses staying high and flat).


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].plot(history_a.history['loss'], label='Train Loss')
axes[0].plot(history_a.history['val_loss'], label='Validation Loss')
axes[0].set_title('Model A: Training vs Validation Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss (MSE, log-price)')
axes[0].legend()

axes[1].plot(history_b.history['loss'], label='Train Loss')
axes[1].plot(history_b.history['val_loss'], label='Validation Loss')
axes[1].set_title('Model B: Training vs Validation Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss (MSE, log-price)')
axes[1].legend()

plt.tight_layout()
plt.show()


**Expected output / note down:**
- For each model, describe in your report whether the validation curve tracks the training curve closely
(good generalisation) or diverges upward (overfitting).
- If Model B (with dropout/batch norm) shows a smaller train/validation gap than Model A, this is direct
evidence that regularisation is helping — a key point for your Task 1 discussion.
- Save/screenshot this plot for your report — it's expected supporting evidence per the marking rubric.


## Step 7: Evaluate Model A & Model B on the Test Set

We evaluate both models on the held-out test set using MAE, RMSE, and R² — the same three metrics
identified in the Assignment 1 literature review (Retnowati et al., 2024) as standard for this type of
comparison. Because the target is log-transformed, we compute metrics **twice**: once in log-space
(directly comparable to the training loss) and once in back-transformed dollar terms (via `np.expm1`),
which is what actually matters to a non-technical reader of the report.

**Design decision:** dollar-space metrics are the primary ones reported in the comparison table and
discussion, since "the model is off by $23,000 on average" is far more meaningful than "off by 0.09
log-units."


In [ ]:
def evaluate_model(model, X, y_true_log, model_name):
    y_pred_log = model.predict(X, verbose=0).flatten()

    # Metrics in log-space
    mae_log = mean_absolute_error(y_true_log, y_pred_log)
    rmse_log = np.sqrt(mean_squared_error(y_true_log, y_pred_log))
    r2_log = r2_score(y_true_log, y_pred_log)

    # Back-transform to dollar terms
    y_true_dollar = np.expm1(y_true_log)
    y_pred_dollar = np.expm1(y_pred_log)

    mae_dollar = mean_absolute_error(y_true_dollar, y_pred_dollar)
    rmse_dollar = np.sqrt(mean_squared_error(y_true_dollar, y_pred_dollar))
    r2_dollar = r2_score(y_true_dollar, y_pred_dollar)

    return {
        'Model': model_name,
        'MAE ($)': mae_dollar, 'RMSE ($)': rmse_dollar, 'R2 ($)': r2_dollar,
        'MAE (log)': mae_log, 'RMSE (log)': rmse_log, 'R2 (log)': r2_log,
    }

results_a = evaluate_model(model_a, X_test, y_test, 'Model A (Baseline)')
results_b = evaluate_model(model_b, X_test, y_test, 'Model B (Deeper + Regularised)')

comparison_df = pd.DataFrame([results_a, results_b]).set_index('Model')
comparison_df


**Expected output / note down:**
- A table with both models' MAE, RMSE, and R² in dollar and log terms.
- Identify which model has the **lower** MAE/RMSE and **higher** R² in dollar terms — this is the
"better-performing model" that Assignment 2, Task 2 explicitly asks you to carry forward into
hyperparameter tuning.
- If the two models perform very similarly, note this explicitly — it's a valid and interesting finding
(echoing Limsombunchai (2004), who found ANN performance often plateaus quickly relative to added
complexity) rather than something to force a conclusion from.


---
# Task 2: Model Tuning

## Step 8: Select the Better-Performing Model

Based on the comparison table in Step 7, identify which model (A or B) will be carried forward for
hyperparameter tuning, as required by the assignment brief. Update the `BEST_MODEL` variable below
after inspecting your actual results.


In [ ]:
# TODO: set this based on your actual Step 7 results (compare dollar-space MAE/RMSE/R2)
BEST_MODEL = 'B'  # change to 'A' if Model A performed better in your run

print(f"Carrying forward Model {BEST_MODEL} for hyperparameter tuning.")


**What to note down:**
- State explicitly in your report *why* this model was selected — cite the specific MAE/RMSE/R² values
from Step 7's comparison table, not just "it performed better."


## Step 9: Define the Hyperparameter Search Space (Keras Tuner)

We use **Keras Tuner** with a `RandomSearch` strategy, as proposed in Assignment 1, to systematically
search over the model's key hyperparameters: number of units per layer, dropout rate, and learning rate.
This directly addresses Task 2's requirement to "list and explain the meta/hyperparameters" of the model
and "choose the optimal parameters using a suitable searching technique."

**Design decision:** the search space below is built around Model B's architecture (assuming it was
selected in Step 8) since it has more tunable components (dropout, multiple layer widths) than Model A.
If Model A was selected instead, remove the dropout-related search parameters and tune only layer widths
and learning rate.


In [ ]:
def build_tunable_model(hp):
    model = keras.Sequential()
    model.add(layers.Input(shape=(n_features,)))

    # Number of hidden layers: 2 to 4
    n_layers = hp.Int('n_layers', min_value=2, max_value=4, step=1)

    for i in range(n_layers):
        units = hp.Choice(f'units_{i}', values=[16, 32, 64, 128])
        model.add(layers.Dense(units, activation='relu'))
        model.add(layers.BatchNormalization())
        dropout_rate = hp.Choice(f'dropout_{i}', values=[0.0, 0.2, 0.3, 0.4])
        model.add(layers.Dropout(dropout_rate))

    model.add(layers.Dense(1))

    learning_rate = hp.Choice('learning_rate', values=[0.01, 0.001, 0.0001])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='mse',
        metrics=['mae']
    )
    return model


**What to note down:**
- List the full search space in your report as a table: `n_layers` [2–4], `units` per layer
[16, 32, 64, 128], `dropout` [0.0, 0.2, 0.3, 0.4], `learning_rate` [0.01, 0.001, 0.0001]. This satisfies
the rubric's requirement to "list and explain" the hyperparameters being tuned.


## Step 10: Run the Hyperparameter Search

We run `RandomSearch` for a fixed number of trials, each evaluated on the validation set. Random search
is used (rather than exhaustive grid search) because the search space above has thousands of possible
combinations, making a full grid search computationally expensive for limited benefit — this trade-off
is worth stating explicitly in your report as a justified choice, not a shortcut.


In [ ]:
tuner = kt.RandomSearch(
    build_tunable_model,
    objective='val_loss',
    max_trials=20,
    executions_per_trial=1,
    directory='keras_tuner_dir',
    project_name='miami_housing_tuning',
    overwrite=True
)

tuner.search_space_summary()


In [ ]:
early_stop_tuning = keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=10, restore_best_weights=True
)

tuner.search(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=32,
    callbacks=[early_stop_tuning],
    verbose=1
)


**Expected output / note down:**
- This step will take noticeably longer to run than a single model fit (20 trials × up to 100 epochs
each, though early stopping will cut most trials short). This is expected — hyperparameter search is
computationally expensive by nature, and this is worth mentioning briefly in your report as a practical
consideration.
- Keras Tuner will print a running summary of each trial's `val_loss` — no need to record every trial
individually, just note the best result once the search completes in Step 11.


## Step 11: Extract Best Hyperparameters & Build Final Tuned Model

We retrieve the best-performing hyperparameter combination found during the search, and use it to build
and train the final tuned model.


In [ ]:
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

print("Best hyperparameters found:")
for param, value in best_hps.values.items():
    print(f"  {param}: {value}")

final_model = tuner.hypermodel.build(best_hps)
final_model.summary()


In [ ]:
early_stop_final = keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=15, restore_best_weights=True
)

history_final = final_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=200,
    batch_size=32,
    callbacks=[early_stop_final],
    verbose=1
)


**Expected output / note down:**
- Record the exact best hyperparameter values printed above — these go directly into your report's Task
2 section as "the optimal hyperparameters found."
- Compare the final tuned model's architecture (number of layers, units) against the untuned Model A/B
from Task 1 — note whether the search converged on something similar to your original design (validating
your original architectural choices) or something quite different (an interesting discussion point).


---
# Task 3: Model Evaluation & Discussion

## Step 12: Evaluate All Three Models on the Test Set

We now evaluate Model A, Model B, and the Final Tuned Model together on the same held-out test set,
producing the three-way comparison required by Task 3.


In [ ]:
results_final = evaluate_model(final_model, X_test, y_test, 'Final Tuned Model')

full_comparison_df = pd.DataFrame([results_a, results_b, results_final]).set_index('Model')
full_comparison_df


**Expected output / note down:**
- A three-row table (Model A, Model B, Final Tuned Model) with MAE/RMSE/R² in both dollar and log terms.
- Check specifically: did tuning **improve** on the better of Model A/B from Task 1? If not, this is a
valid and worth-discussing finding — Retnowati et al. (2024) similarly found that a more complex model
(ANN) did not always outperform a simpler alternative (Random Forest) in their comparison, so an
unexpected result here has literature precedent and is not necessarily something to hide.


## Step 13: Visual Comparison — Metrics Bar Chart

A bar chart makes the three-way comparison from Step 12 easier to interpret at a glance than the raw
table alone, and is good, low-effort supporting evidence for the report.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

full_comparison_df[['MAE ($)', 'RMSE ($)']].plot(kind='bar', ax=axes[0])
axes[0].set_title('MAE and RMSE by Model (Dollar Terms)')
axes[0].set_ylabel('Error ($)')
axes[0].tick_params(axis='x', rotation=15)

full_comparison_df[['R2 ($)']].plot(kind='bar', ax=axes[1], legend=False, color='seagreen')
axes[1].set_title('R² by Model (Dollar Terms)')
axes[1].set_ylabel('R²')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()


**Expected output / note down:**
- Save this chart directly into your report as the primary visual evidence for Task 3.


## Step 14: Predicted vs. Actual Price Scatter Plot (Final Model)

A scatter plot of predicted vs. actual prices for the final model reveals *where* the model struggles —
e.g. systematically underpredicting very expensive properties — which a single aggregate metric like
RMSE cannot show on its own.


In [ ]:
y_pred_final_log = final_model.predict(X_test, verbose=0).flatten()
y_pred_final_dollar = np.expm1(y_pred_final_log)
y_test_dollar = np.expm1(y_test)

plt.figure(figsize=(8, 8))
plt.scatter(y_test_dollar, y_pred_final_dollar, alpha=0.4, s=15)
max_val = max(y_test_dollar.max(), y_pred_final_dollar.max())
plt.plot([0, max_val], [0, max_val], 'r--', label='Perfect Prediction')
plt.xlabel('Actual Sale Price ($)')
plt.ylabel('Predicted Sale Price ($)')
plt.title('Predicted vs. Actual Sale Price — Final Tuned Model')
plt.legend()
plt.tight_layout()
plt.show()


**Expected output / note down:**
- Points should cluster fairly tightly around the red diagonal line for a well-performing model.
- Specifically check the **upper-right region** (high-value properties): if points fall noticeably below
the diagonal there, the model is **underpredicting expensive homes** — a common pattern in price
prediction, worth flagging explicitly as a limitation in Task 4.


## Step 15: Residual Analysis

We plot residuals (actual minus predicted) against predicted price to check for patterns — ideally,
residuals should scatter randomly around zero with no obvious trend. A funnel shape (residual spread
increasing with price) would indicate the model is less reliable for expensive properties.


In [ ]:
residuals = y_test_dollar - y_pred_final_dollar

plt.figure(figsize=(10, 6))
plt.scatter(y_pred_final_dollar, residuals, alpha=0.4, s=15)
plt.axhline(0, color='red', linestyle='--')
plt.xlabel('Predicted Sale Price ($)')
plt.ylabel('Residual (Actual - Predicted, $)')
plt.title('Residual Plot — Final Tuned Model')
plt.tight_layout()
plt.show()

print(f"Mean residual: ${residuals.mean():,.0f}")
print(f"Std of residuals: ${residuals.std():,.0f}")


**Expected output / note down:**
- Note whether residuals fan out (increase in spread) at higher predicted prices — this is common in
real estate price prediction and should be discussed as a limitation rather than treated as a bug.
- A mean residual close to $0 indicates the model is not systematically biased high or low overall, even
if individual predictions vary.


---
# Task 4: Conclusion

## Step 16: Guidance for the Written Conclusion

This step is not code — it's a checklist for the written conclusion required by Task 4, using the
outputs generated above.

**Your conclusion should cover:**

1. **Best-performing model and why.** State which of the three models (A, B, Final Tuned) achieved the
best test-set metrics (Step 12), quoting the actual MAE/RMSE/R² values in dollar terms.

2. **Comparison against the literature review (Section 5 of Assignment 1).** For example:
   - Does your finding that `TOT_LVG_AREA` and locational distance features drive predictions align with
     the hedonic pricing theory reviewed (Usman, 2020)?
   - Did tuning improve results, consistent with the general expectation set by Limsombunchai (2004) and
     Feng and Jones (2015) that ANNs can match/exceed simpler baselines? Or did you find, like Retnowati
     et al. (2024), that added model complexity did not guarantee better performance?

3. **Limitations**, informed directly by what you observed in Steps 6, 14, and 15:
   - Any overfitting/underfitting signs seen in the training curves.
   - Any systematic underprediction of high-value properties seen in the scatter/residual plots.
   - The dataset covers only a single year (2016) and a single city (Miami) — results are not
     necessarily generalisable to other markets or time periods.
   - The `avno60plus` feature's extreme imbalance (noted in Assignment 1) limited its practical
     predictive contribution despite its theoretical relevance.

4. **Future improvements**, which can reference Hasan et al. (2024) from the literature review:
   incorporating additional data modalities (property images, textual listing descriptions, or
   neighbourhood-level context) beyond the purely tabular features used here, or extending the dataset
   across multiple years to capture market trends over time.
